# Task 1 — SQL Query Performance Analysis

This project focuses on analyzing and optimizing SQL queries
for better performance using Microsoft SQL Server.

## Task Description

Analyze both given scripts and propose a way for improvement
in terms of performance.

Please describe in detail what is(are) a problem(s) and how it
is possible to solve it (them). Try to rewrite the code in the
right manner.

The logical models are in attached files.

For the first script, try to show at least 2 solutions
(for example, using regular SQL and CTE / using temporary
tables / using window functions, etc.), suppose which one is
(or would be) better and why.

Don't forget that SQL Server creates indexes automatically
for primary keys and alternate keys specified as unique
constraints.

# 1. First Script

## 1.1 Database Logical Model

The following logical model represents the database used
for the first script.

![First Database Logical Model](images/first-model.png)

## 1.2 Original Script

The following query is the original script provided in the task.

In [5]:
USE meteo_sandbox_db
GO

-- For each GeonameSubclass which includes 'mountain' in the name at any position and belongs to GeonameClass 'T', show :
-- * all corresponding names of mountains (Geoobjects) which are shared between at least 2 countries, 
-- * the kind of mountain (field name in GeonameSubclass table),
-- * that country names, 
-- * and the number of countries which are shared the object
-- Example of result:
-- national_name       kind_of_mountains  number_of_shared_country shared_country
-- Alí Boutoús         mountain           2                        Greece
-- Alí Boutoús         mountain           2                        Bulgaria
-- Anatolikí Rodhópi   mountains          2                        Greece

SELECT gobj.national_name, 
    (SELECT [name] FROM dbo.GeonameSubclass AS gsc WHERE gsc.geonameSubclassId = gobj.geonameSubclassId) AS kind_of_mountains,
    (SELECT count(*) FROM dbo.SharedCountry AS sc WHERE sc.geoobjectId = gobj.geoobjectId) AS number_of_shared_country,
    c.[name] AS shared_country
FROM dbo.Geoobject AS gobj
LEFT JOIN dbo.SharedCountry AS sc ON sc.geoobjectId = gobj.geoobjectId
LEFT JOIN dbo.Country AS c ON sc.countryId = c.countryId
WHERE gobj.geonameSubclassId IN 
    (
        SELECT gsc.geonameSubclassId
        FROM dbo.GeonameClass AS gc
        JOIN dbo.GeonamesubClass AS gsc ON gc.geonameClassId = gsc.geonameClassId
        WHERE gc.code = 'T' AND gsc.[name] LIKE '%mountain%'
    )
    AND
    (SELECT count(*) FROM dbo.SharedCountry AS sc WHERE sc.geoobjectId = gobj.geoobjectId) > 1
ORDER BY gobj.national_name

Commands completed successfully.

(82 rows affected)

national_name         | kind_of_mountains | number_of_shared_country | shared_country 
----------------------+-------------------+--------------------------+----------------
Alí Boutoús           | mountain          | 2                        | Greece         
Alí Boutoús           | mountain          | 2                        | Bulgaria       
Anatolikí Rodhópi     | mountains         | 2                        | Greece         
Anatolikí Rodhópi     | mountains         | 2                        | Bulgaria       
Bakŭrchala            | mountain          | 2                        | Bulgaria       
Bakŭrchala            | mountain          | 2                        | Greece         
Góry Bialskie         | mountains         | 2                        | Czechia        
Góry Bialskie         | mountains         | 2                        | Poland         
Gradište              | mountain          | 2                        | Serbi

## 1.3 Initial Analysis

The initial query is functionally correct, but it has several disadvantages in terms of performance. The main problem is the use of correlated subqueries that depend on the current row of the Geoobject table. The number of countries associated with a Geoobject is also calculated twice: once in the SELECT clause and once in the WHERE clause. This can lead to repeated work. The use of LEFT JOIN is also unnecessary because the required result contains only Geoobject records that have corresponding countries and are shared between at least two countries, so INNER JOIN is more appropriate.

Three improved solutions were considered. The first solution uses a CTE and the window function OVER(PATRITION BY geoobjectId). This eliminates the repeated correlated count and provides the number of countries for every Geoobject. The second solution uses a CTE with GROUP BY and HAVING COUNT(*) > 1. This directly selects only those Geoobject records that are shared between at least two countries. The third solution uses temporary tables. This allows intermediate results to be materialized and indexed, which can be useful for large intermediate datasets or when the same results are used several times. However, temporary tables introduce additional tempdb operations, so they are not necessarily more efficient for this query. 
The CTE with GROUP BY and HAVING is considered the most appropriate solution for this particular task. It has a simple and readable structure, directly represents the condition that an object must be shared between at least two countries, and does not require additional temporary tables. The window-function solution is also efficient, while the temporary-table solution is more suitable for more complex queries or large intermediate results.

The condition LIKE '%mountain%' is another limitation. Because the pattern starts with %, a conventional B-tree index on the name column cannot normally perform an efficient index seek for this condition. It cannot be changed to LIKE 'mountain%', because the task requires finding mountain at any position in the name.

When creating additional indexes, the indexes that SQL Server already creates for primary keys and alternate keys must be taken into account. GeonameClass.code is an alternate key, so it already has a unique index. Therefore, an additional index on GeonameClass(code) is unnecessary. Country.countryId is a primary key, so it is also already indexed and does not require another index. The SharedCountry table has a composite primary key (geoobjectId, countryId). Since this index starts with geoobjectId, it can already be used efficiently for searches, joins and grouping by geoobjectId, so an additional index only on SharedCountry(geoobjectId) is generally redundant.

Two additional indexes are useful for this query. The first index should be created on GeonameSubclass(geonameClassId) with name as an included column. This index supports the join between GeonameClass and GeonameSubclass and provides the name value needed to identify the kind of mountain. The second index should be created on Geoobject(geonameSubclassId) with national_name as an included column. This index supports the join between the selected subclasses and Geoobject and provides the national_name required in the result. The primary key columns do not need to be explicitly included because, according to the conditions of the task, the primary-key indexes already contain the primary-key columns.

Therefore, the additional indexes that should be created are:

IX_GeonameSubclass_GeonameClassId on GeonameSubclass(geonameClassId) with name included — to efficiently find subclasses belonging to the selected GeonameClass and obtain their names.

IX_Geoobject_GeonameSubclassId on Geoobject(geonameSubclassId) with national_name included — to efficiently find the required Geoobject records by their subclass and obtain their names.

Indexes on GeonameClass(code), Country(countryId) and SharedCountry(geoobjectId) should not be added because they would duplicate or be largely covered by the indexes that already exist due to the alternate and primary keys.

Overall, the optimized queries reduce repeated calculations and unnecessary joins, while the additional indexes support the main join operations. For this particular task, the CTE with GROUP BY and HAVING provides the best balance between performance, simplicity and readability.

## 1.4 Solution 1 — CTE + GROUP BY / HAVING

In [6]:
USE meteo_sandbox_db;
GO

WITH MountainSubclasses AS
(
    SELECT
        gsc.geonameSubclassId,
        gsc.name AS kind_of_mountains
    FROM dbo.GeonameClass AS gc
    INNER JOIN dbo.GeonameSubclass AS gsc
        ON gsc.geonameClassId = gc.geonameClassId
    WHERE gc.code = 'T'
      AND gsc.name LIKE '%mountain%'
),
SharedCounts AS
(
    SELECT
        sc.geoobjectId,
        COUNT(*) AS number_of_shared_country
    FROM dbo.SharedCountry AS sc
    GROUP BY
        sc.geoobjectId
    HAVING COUNT(*) > 1
)
SELECT
    gobj.national_name,
    ms.kind_of_mountains,
    scnt.number_of_shared_country,
    c.name AS shared_country
FROM dbo.Geoobject AS gobj
INNER JOIN MountainSubclasses AS ms
    ON ms.geonameSubclassId = gobj.geonameSubclassId
INNER JOIN SharedCounts AS scnt
    ON scnt.geoobjectId = gobj.geoobjectId
INNER JOIN dbo.SharedCountry AS sc
    ON sc.geoobjectId = gobj.geoobjectId
INNER JOIN dbo.Country AS c
    ON c.countryId = sc.countryId
ORDER BY
    gobj.national_name,
    c.name;
GO

Commands completed successfully.

(82 rows affected)

national_name         | kind_of_mountains | number_of_shared_country | shared_country 
----------------------+-------------------+--------------------------+----------------
Alí Boutoús           | mountain          | 2                        | Bulgaria       
Alí Boutoús           | mountain          | 2                        | Greece         
Anatolikí Rodhópi     | mountains         | 2                        | Bulgaria       
Anatolikí Rodhópi     | mountains         | 2                        | Greece         
Bakŭrchala            | mountain          | 2                        | Bulgaria       
Bakŭrchala            | mountain          | 2                        | Greece         
Góry Bialskie         | mountains         | 2                        | Czechia        
Góry Bialskie         | mountains         | 2                        | Poland         
Gradište              | mountain          | 2                        | North

## 1.5 Solution 2 — CTE + Window Function

In [9]:
USE meteo_sandbox_db;
GO

WITH MountainSubclasses AS
(
    SELECT
        gsc.geonameSubclassId,
        gsc.name AS kind_of_mountains
    FROM dbo.GeonameClass AS gc
    INNER JOIN dbo.GeonameSubclass AS gsc
        ON gsc.geonameClassId = gc.geonameClassId
    WHERE gc.code = 'T'
      AND gsc.name LIKE '%mountain%'
),
SharedWithCount AS
(
    SELECT
        sc.geoobjectId,
        sc.countryId,
        COUNT(*) OVER
        (
            PARTITION BY sc.geoobjectId
        ) AS number_of_shared_country
    FROM dbo.SharedCountry AS sc
)
SELECT
    gobj.national_name,
    ms.kind_of_mountains,
    swc.number_of_shared_country,
    c.name AS shared_country
FROM dbo.Geoobject AS gobj
INNER JOIN MountainSubclasses AS ms
    ON ms.geonameSubclassId = gobj.geonameSubclassId
INNER JOIN SharedWithCount AS swc
    ON swc.geoobjectId = gobj.geoobjectId
INNER JOIN dbo.Country AS c
    ON c.countryId = swc.countryId
WHERE swc.number_of_shared_country > 1
ORDER BY
    gobj.national_name,
    c.name;
GO

Commands completed successfully.

(82 rows affected)

national_name         | kind_of_mountains | number_of_shared_country | shared_country 
----------------------+-------------------+--------------------------+----------------
Alí Boutoús           | mountain          | 2                        | Bulgaria       
Alí Boutoús           | mountain          | 2                        | Greece         
Anatolikí Rodhópi     | mountains         | 2                        | Bulgaria       
Anatolikí Rodhópi     | mountains         | 2                        | Greece         
Bakŭrchala            | mountain          | 2                        | Bulgaria       
Bakŭrchala            | mountain          | 2                        | Greece         
Góry Bialskie         | mountains         | 2                        | Czechia        
Góry Bialskie         | mountains         | 2                        | Poland         
Gradište              | mountain          | 2                        | North

## 1.6 Solution 3 — Temporary Tables

In [10]:
USE meteo_sandbox_db;
GO

IF OBJECT_ID('tempdb..#MountainGeoobjects') IS NOT NULL
    DROP TABLE #MountainGeoobjects;

IF OBJECT_ID('tempdb..#SharedCounts') IS NOT NULL
    DROP TABLE #SharedCounts;


SELECT
    gobj.geoobjectId,
    gobj.national_name,
    gobj.geonameSubclassId,
    gsc.name AS kind_of_mountains
INTO #MountainGeoobjects
FROM dbo.Geoobject AS gobj
INNER JOIN dbo.GeonameSubclass AS gsc
    ON gsc.geonameSubclassId = gobj.geonameSubclassId
INNER JOIN dbo.GeonameClass AS gc
    ON gc.geonameClassId = gsc.geonameClassId
WHERE gc.code = 'T'
  AND gsc.name LIKE '%mountain%';


CREATE CLUSTERED INDEX IX_tmp_MountainGeoobjects_geoobjectId
ON #MountainGeoobjects(geoobjectId);

SELECT
    sc.geoobjectId,
    COUNT(*) AS number_of_shared_country
INTO #SharedCounts
FROM dbo.SharedCountry AS sc
INNER JOIN #MountainGeoobjects AS mg
    ON mg.geoobjectId = sc.geoobjectId
GROUP BY
    sc.geoobjectId
HAVING COUNT(*) > 1;

CREATE CLUSTERED INDEX IX_tmp_SharedCounts_geoobjectId
ON #SharedCounts(geoobjectId);


SELECT
    mg.national_name,
    mg.kind_of_mountains,
    scnt.number_of_shared_country,
    c.name AS shared_country
FROM #MountainGeoobjects AS mg
INNER JOIN #SharedCounts AS scnt
    ON scnt.geoobjectId = mg.geoobjectId
INNER JOIN dbo.SharedCountry AS sc
    ON sc.geoobjectId = mg.geoobjectId
INNER JOIN dbo.Country AS c
    ON c.countryId = sc.countryId
ORDER BY
    mg.national_name,
    c.name;
GO

Commands completed successfully.

(2448 rows affected)
(41 rows affected)
(82 rows affected)

national_name         | kind_of_mountains | number_of_shared_country | shared_country 
----------------------+-------------------+--------------------------+----------------
Alí Boutoús           | mountain          | 2                        | Bulgaria       
Alí Boutoús           | mountain          | 2                        | Greece         
Anatolikí Rodhópi     | mountains         | 2                        | Bulgaria       
Anatolikí Rodhópi     | mountains         | 2                        | Greece         
Bakŭrchala            | mountain          | 2                        | Bulgaria       
Bakŭrchala            | mountain          | 2                        | Greece         
Góry Bialskie         | mountains         | 2                        | Czechia        
Góry Bialskie         | mountains         | 2                        | Poland         
Gradište              | mountain    

# 2. Second Script

## 2.1 Database Logical Model

The following logical model represents the database used
for the first script.

![Second Database Logical Model](images/second-model.png)

## 2.2 Original Script

The following query is the original script provided in the task.

In [11]:
USE meteo_sandbox_db
GO

-- Find out the maximal difference between mininum and maximum of temperature in the same point (GeoPoint) 
-- on January 1st at 00:00 for different years, show this maximal difference, longitude and latitude of corresponding point.

DECLARE 
    @minId INT = (SELECT min(geopointId) FROM dbo.GeoPoint), -- minimal geopointId
    @maxId INT = (SELECT max(geopointId) FROM dbo.GeoPoint), -- maximal geopointId
    @curId INT,             -- geopointId for current point
    @lat FLOAT,             -- latitude of GeoPoint with maximal temperature difference
    @long FLOAT,            -- longitude of GeoPoint with maximal temperature difference
    @curdif FLOAT,          -- the maximal temperature difference for current GeoPoint
    @maxdif FLOAT = 0.0     -- the maximal temperature difference throughout all points
    
SET @curId = @minId

WHILE (@curId <= @maxId)
BEGIN

    SELECT @curdif = max(m.temperature) - min(m.temperature) 
    FROM dbo.Measurement AS m 
    INNER JOIN dbo.TimePoint AS tp ON m.timePointId = tp.timePointId
    WHERE m.geopointId = @curId AND tp.[month] = 1 AND tp.[day] = 1 AND tp.[hour] = 0 AND tp.[minute] = 0

    IF @curdif > @maxdif
    BEGIN
        SET @maxdif = @curdif

        SELECT @lat = latitude, @long = longitude 
        FROM dbo.GeoPoint
        WHERE geoPointId = @curId
    END
    SET @curId = @curId + 1
END
SELECT @lat AS latitude, @long AS longitude, @maxdif AS mxdt


Commands completed successfully.

(1 row affected)

latitude | longitude | mxdt              
---------+-----------+-------------------
52.37    | 9.74      | 13.600000000000001
(1 row)

Total execution time: 00:00:00.384

## 2.3 Initial Analysis

Problem: The original query uses row-by-row processing with a WHILE loop. For every GeoPoint, a separate aggregation over the related measurements is performed. This introduces additional procedural overhead and can cause repeated access to the Measurement and TimePoint data. The performance becomes worse as the number of GeoPoints increases.

Problem: The query uses scalar variables together with IF and SET statements to store the current temperature difference and the maximum value. This makes the query procedural instead of set-based. The same task can be performed more efficiently using SQL aggregation functions.

Problem: The loop iterates through every integer value between the minimum and maximum geoPointId. If some IDs are missing, the query still checks these values even though there is no corresponding GeoPoint. Therefore, the loop performs unnecessary iterations.

Solution: The WHILE loop can be eliminated by using a CTE with GROUP BY. The query first selects the required measurements for January 1st at 00:00 for all years. Then MAX(temperature) - MIN(temperature) is calculated for each GeoPoint. Finally, TOP 1 with ORDER BY is used to select the point with the maximum temperature difference.

This set-based approach allows SQL Server to process the data as a single relational operation instead of executing the same logic separately for every GeoPoint. It also eliminates the scalar variables and procedural IF/SET operations from the original query.

The additional index IX_TimePoint_Date is created on TimePoint using month, day, hour, and minute as key columns and including timePointId. This index supports the filtering conditions used to find January 1st at 00:00 and also provides the timePointId required for the join with Measurement.

The additional index IX_Measurement_Time_Geo is created on Measurement using timePointId and geoPointId as key columns and including temperature. timePointId is used for joining Measurement with TimePoint, geoPointId is used for grouping the measurements by point, and temperature is required for the MAX and MIN calculations. Therefore, this index can provide all the required columns for this part of the query without unnecessary lookups.

No additional index is created on GeoPoint(geoPointId) because geoPointId is already a primary key. According to the conditions of the task, SQL Server automatically creates an index for the primary key, so creating another index on the same column would be redundant. The existing primary-key index is sufficient for the final join with GeoPoint and for retrieving the corresponding latitude and longitude.

The ORDER BY t.temp_diff DESC, t.geoPointId is used to select the maximum temperature difference. The additional geoPointId ordering also makes the result deterministic if several points have the same maximum temperature difference.

Conclusion: The optimized CTE solution is more efficient than the original WHILE-based implementation because it eliminates row-by-row processing, scalar variable assignments, and repeated per-point aggregation. The required temperature difference is calculated for all relevant GeoPoints using one set-based aggregation, and TOP 1 is then used to find the maximum value. The IX_TimePoint_Date and IX_Measurement_Time_Geo indexes support the main filtering, joining, grouping, and aggregation operations. An additional index on GeoPoint is not required because the primary key is already indexed automatically. Therefore, the optimized solution is simpler, more readable, and better suited for processing a large number of GeoPoints and measurements.

## 2.4 Solution 

In [12]:
USE meteo_sandbox_db;
GO

WITH TempDiff AS
(
    SELECT
        m.geoPointId,
        MAX(m.temperature) - MIN(m.temperature) AS temp_diff
    FROM dbo.Measurement AS m
    INNER JOIN dbo.TimePoint AS tp
        ON m.timePointId = tp.timePointId
    WHERE tp.[month] = 1
      AND tp.[day] = 1
      AND tp.[hour] = 0
      AND tp.[minute] = 0
    GROUP BY m.geoPointId
)
SELECT TOP 1
    g.latitude,
    g.longitude,
    t.temp_diff AS max_temperature_difference
FROM TempDiff AS t
INNER JOIN dbo.GeoPoint AS g
    ON t.geoPointId = g.geoPointId
ORDER BY
    t.temp_diff DESC,
    t.geoPointId;
GO

Commands completed successfully.

(1 row affected)

latitude | longitude | max_temperature_difference
---------+-----------+---------------------------
52.37    | 9.74      | 13.600000000000001        
(1 row)

Total execution time: 00:00:00.022